### Margin decline analysis

We observe the following pattern in our data: the profit margin is declining while the business is growing.

In this notebook, we will break down this trend to identify what has changed: prices, costs, discounts, or sales mix.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
input_file = "../data/dear_neighbour.xlsx"

sheets = pd.read_excel(input_file, sheet_name=None)
sheet_names = sheets.keys()

sheet_names

dict_keys(['fact_sales', 'dim_products', 'dim_stores', 'dim_customers', 'dim_suppliers'])

In [26]:
sales = sheets['fact_sales']
sales.sample()

,receipt_id,datetime,store_id,product_id,customer_id,quantity,unit_price,discount_percent,line_total,payment_method
49386,1013996,2025-04-02 12:38:00,2,69,2656.0,1,94.18,0,94.18,card


In [21]:
products = sheets['dim_products']
products.sample()

,product_id,product_name,brand,category,subcategory,purchase_price_rub,retail_price_rub,supplier_id
0,1,milk 3.2% 1l,lugovoye,dairy products,milk,64.46,82.64,1


[As we saw](https://github.com/TAbramovskaya/dear-neighbour/blob/main/python/data_overview.ipynb), selling prices remained relatively stable over the two-year period.

Therefore, we will overlook the fact that we have a single purchase price for each product across both years.

In [27]:
sales['year'] = sales['datetime'].dt.year

sales = (
    sales.merge(
        products[['product_id', 'category', 'subcategory', 'purchase_price_rub']],
        on='product_id',
        how='left'
    )
)

sales['unit_revenue'] = sales['line_total'] / sales['quantity']

sales['cost'] = sales['purchase_price_rub'] * sales['quantity']

sales['gross_profit'] = sales['line_total'] - sales['cost']

In [28]:
sales.sample()

,receipt_id,datetime,store_id,product_id,customer_id,quantity,unit_price,discount_percent,line_total,payment_method,year,category,subcategory,purchase_price_rub,unit_revenue,cost,gross_profit
79456,1022311,2025-12-17 17:17:00,6,81,127.0,1,109.58,10,98.62,card,2025,alcohol,beer,79.54,98.62,79.54,19.08


#### Compare overall and category-level margins by year

In [30]:
summary_by_year = (
    sales
    .groupby(['year'])
    .agg(
        revenue=('line_total', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        costs=('cost', 'sum'),
        quantity=('quantity', 'sum'),
        receipts=('receipt_id', 'nunique')
    )
)

In [33]:
summary_by_year['gross_margin'] = summary_by_year['gross_profit'] / summary_by_year['revenue']

summary_by_year['avg_receipt'] = summary_by_year['revenue'] / summary_by_year['receipts']

In [34]:
summary_by_year

,revenue,gross_profit,costs,quantity,receipts,gross_margin,avg_receipt
year,,,,,,,
2024,10432017.93,2751014.93,7681003.00,50334,11019,0.263709,946.730005
2025,11457582.40,3008785.06,8448797.34,55196,11771,0.262602,973.373749


In [36]:
summary_by_category = (
    sales
    .groupby(['year', 'category'])
    .agg(
        revenue=('line_total', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        costs=('cost', 'sum'),
        quantity=('quantity', 'sum')
    )
)

In [37]:
summary_by_category['gross_margin'] = summary_by_category['gross_profit'] / summary_by_category['revenue']

summary_by_category['revenue_share'] = (
        summary_by_category['revenue']
        / summary_by_category.groupby(level=['year'])['revenue'].transform('sum')
)

In [38]:
summary_by_category

revenue  gross_profit       costs  quantity  \
year category                                                                
2024 alcohol                1359618.65     362576.67   997041.98      3846   
     baby food               376880.41     109120.51   267759.90      3511   
     beverages               808468.50     243861.68   564606.82      3790   
     bread and bakery        194230.17      55887.71   138342.46      3290   
     confectionery           533736.62     170658.80   363077.82      3512   
     dairy products          441188.73     121022.70   320166.03      3634   
     fish and seafood       1823379.98     436317.07  1387062.91      3524   
     frozen products         681856.98     184179.96   497677.02      3664   
     groceries               539581.77     124720.26   414861.51      3727   
     meat and poultry       1329415.58     259703.04  1069712.54      3533   
     non-food products       774059.12     194271.63   579787.49      3463   
     ready-made food         655152.29     236900.98   418251.31      3605   
     snacks                  551359.25     171740.38   379618.87      3640   
     vegetables and fruits   363089.88      80053.54   283036.34      3595   
2025 alcohol                1412617.21     373823.91  1038793.30      4010   
     baby food               416661.74     120411.12   296250.62      3867   
     beverages               918899.53     278872.23   640027.30      4188   
     bread and bakery        223043.46      64274.92   158768.54      3784   
     confectionery           579876.34     184622.08   395254.26      3876   
     dairy products          460810.59     125899.21   334911.38      3799   
     fish and seafood       2094865.50     496721.56  1598143.94      4072   
     frozen products         758083.13     206014.53   552068.60      4054   
     groceries               570211.38     132017.38   438194.00      3912   
     meat and poultry       1454899.53     282577.81  1172321.72      3900   
     non-food products       865515.10     216580.61   648934.49      3817   
     ready-made food         698030.78     252168.59   445862.19      3836   
     snacks                  587133.70     182952.35   404181.35      3956   
     vegetables and fruits   416934.41      91848.76   325085.65      4125   

                            gross_margin  revenue_share  
year category                                            
2024 alcohol                    0.266675       0.130331  
     baby food                  0.289536       0.036127  
     beverages                  0.301634       0.077499  
     bread and bakery           0.287740       0.018619  
     confectionery              0.319743       0.051163  
     dairy products             0.274310       0.042292  
     fish and seafood           0.239290       0.174787  
     frozen products            0.270115       0.065362  
     groceries                  0.231142       0.051724  
     meat and poultry           0.195351       0.127436  
     non-food products          0.250978       0.074200  
     ready-made food            0.361597       0.062802  
     snacks                     0.311485       0.052853  
     vegetables and fruits      0.220479       0.034805  
2025 alcohol                    0.264632       0.123291  
     baby food                  0.288990       0.036366  
     beverages                  0.303485       0.080200  
     bread and bakery           0.288172       0.019467  
     confectionery              0.318382       0.050611  
     dairy products             0.273212       0.040219  
     fish and seafood           0.237114       0.182837  
     frozen products            0.271757       0.066164  
     groceries                  0.231524       0.049767  
     meat and poultry           0.194225       0.126981  
     non-food products          0.250233       0.075541  
     ready-made food            0.361257       0.060923  
     snacks                     0.311603       0.051244  
     veg

#### Margin Decomposition: Mix Effect vs. Within-Category Effect

Overall gross margin is a weighted average of category-level gross margins:

$$ M_t = \sum_c s^t_c m^t_c$$

where:
- $M_t$ — overall gross margin in year $t$.
- $s^t_c$ — category $c$'s share of total revenue in year $t$.
- $m^t_c$ — category $c$'s gross margin in year $t$.

Therefore, overall gross margin depends on two factors: *how total revenue is distributed across categories* and *how profitable each category is*.


Say, $t$ is equal to 1 for 2024, and 2 for 2025. The total change in gross margin between two years is:

$$\Delta M = M_2 - M_1 = \sum_c s^2_c m^2_c - \sum_c s^1_c m^1_c$$

We want to separate this change into two effects:

- **Mix effect:** the impact of changes in category revenue shares.
- **Within-category margin effect:** the impact of changes in category-level margins.

Both category revenue shares and category margins may change simultaneously. To separate their effects, we use a symmetric decomposition:

$$\Delta M_{\text{mix}} = \sum_c (s^2_c-s^1_c) \frac{m^1_c+m^2_c}{2}$$

$$\Delta M_{\text{margin}} = \sum_c (m^2_c-m^1_c) \frac{s^1_c+s^2_c}{2}$$

The two effects add up exactly to the observed change in overall gross margin:

$$\Delta M = \Delta M_{\text{mix}} + \Delta M_{\text{margin}}$$

#### Interpretation

- A **negative mix effect** means that changes in revenue shares reduced overall gross margin. For example, lower-margin categories may have gained revenue share.
- A **negative within-category margin effect** means that category-level margins declined, reducing overall gross margin.
- A positive effect means that the corresponding changes contributed to an increase in overall gross margin.

Both effects can be positive or negative, and they can offset each other.

The results are expressed in gross-margin proportions. Multiply them by 100 to express the effects in **percentage points (pp)**.

This decomposition helps identify whether the decline in overall gross margin is primarily associated with changes in sales mix or declining profitability within categories. It is an accounting decomposition, not proof of causality.

In [47]:
summary_by_category = summary_by_category.reset_index()

In [50]:
(
    summary_by_category
    .loc[summary_by_category['year'] == 2024]
    .assign(
        weighted_margin=lambda x: x['gross_margin'] * x['revenue_share']
    )['weighted_margin']
    .sum()
)

np.float64(0.2637088000097024)

In [53]:
shares = summary_by_category.pivot(
    index='category',
    columns='year',
    values='revenue_share'
)

shares.sample(2)

year,2024,2025
category,,
meat and poultry,0.127436,0.126981
beverages,0.077499,0.080200


In [54]:
margins = summary_by_category.pivot(
    index='category',
    columns='year',
    values='gross_margin'
)

margins.sample(2)

year,2024,2025
category,,
beverages,0.301634,0.303485
ready-made food,0.361597,0.361257


In [59]:
t1 = 2024
t2 = 2025

comparison = pd.DataFrame(index=shares.index)

comparison['mix_effect_pp'] = (100 * (shares[t2] - shares[t1]) * (margins[t1] + margins[t2]) / 2)

comparison['margin_effect_pp'] = (100 * (margins[t2] - margins[t1]) * (shares[t1] + shares[t2]) / 2)

comparison['total_effect_pp'] = (comparison['mix_effect_pp'] + comparison['margin_effect_pp'])

comparison.sort_values('total_effect_pp')

,mix_effect_pp,margin_effect_pp,total_effect_pp
category,,,
alcohol,-0.187028,-0.025909,-0.212937
ready-made food,-0.067913,-0.002101,-0.070014
dairy products,-0.056750,-0.004530,-0.061279
snacks,-0.050111,0.000609,-0.049502
groceries,-0.045259,0.001934,-0.043325
confectionery,-0.017632,-0.006929,-0.024561
meat and poultry,-0.008857,-0.014328,-0.023185
baby food,0.006893,-0.001979,0.004914
bread and bakery,0.024425,0.000824,0.025249
